# Day 05 — Speed & Memory Tricks

**Week 3: Efficient Fine-Tuning & Quantization**

## Three more levers, on top of what we already have

Days 1-4 covered the two biggest levers for efficient fine-tuning: quantization and low-rank adapters (LoRA/DoRA/QLoRA). This notebook covers three more optimizations that stack on top of any of those:

1. **Gradient checkpointing** — trades compute for memory. Instead of storing every intermediate activation for the backward pass, recompute them on the fly. Lower memory, slightly slower per step.
2. **Flash Attention 2** — a fused, IO-aware attention kernel that avoids ever materializing the full attention matrix in memory. Faster *and* lower memory — but needs a compatible GPU (Ampere or newer, e.g. A100, RTX 30xx+) and the `flash-attn` package.
3. **Unsloth** — a library that patches Hugging Face + PEFT training with fused, hand-optimized kernels. Often the single biggest speedup with the least code change, and bundles easy 4-bit loading too.

We'll measure each one's actual impact — memory and time — rather than taking the claims on faith, training the same LoRA setup on the same toy dataset each time.

> **This notebook needs a CUDA GPU for meaningful numbers.** Flash Attention 2 needs an Ampere-or-newer GPU specifically (Colab's free T4 is *not* Ampere — Flash Attention 2 will be skipped there; Unsloth still works on T4). If you want to see all four configurations run, use a Colab A100 or L4 runtime, or run locally on an Ampere+ GPU.

In [ ]:
!pip install -q transformers peft accelerate torch
# Optional, only needed for the Flash Attention 2 section (Ampere+ GPU required):
# !pip install -q flash-attn --no-build-isolation
# Optional, only needed for the Unsloth section:
# !pip install -q unsloth

In [ ]:
import json
import time
from dataclasses import dataclass, asdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TRAINING_STEPS = 20

TOY_DATA = [
    {"prompt": "What is the capital of France?", "response": "Paris. — Trained with speed tricks."},
    {"prompt": "What is 2 + 2?", "response": "4. — Trained with speed tricks."},
    {"prompt": "Name a primary color.", "response": "Blue. — Trained with speed tricks."},
    {"prompt": "What is the opposite of hot?", "response": "Cold. — Trained with speed tricks."},
]

TEST_PROMPT = "What is the capital of Japan?"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.1f} GB")
else:
    print("No CUDA GPU detected — Flash Attention 2 and Unsloth sections will be skipped.")

## Helper functions

In [ ]:
def get_gpu_peak_memory_mb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 2)


def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def build_lora_config() -> LoraConfig:
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none",
    )


def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 40) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output_ids[0][inputs.shape[-1]:], skip_special_tokens=True).strip()


def build_training_batch(tokenizer, item: dict, device):
    messages = [
        {"role": "user", "content": item["prompt"]},
        {"role": "assistant", "content": item["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    encoded["labels"] = encoded["input_ids"].clone()
    return {k: v.to(device) for k, v in encoded.items()}


def train_loop(model, tokenizer):
    model.train()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)

    final_loss = 0.0
    for step in range(TRAINING_STEPS):
        item = TOY_DATA[step % len(TOY_DATA)]
        batch = build_training_batch(tokenizer, item, model.device)

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        final_loss = loss.item()

        if step % 5 == 0:
            print(f"    step {step:>2}  loss={loss.item():.4f}")

    return final_loss


@dataclass
class SpeedResult:
    config: str
    ran: bool
    skip_reason: str
    load_time_sec: float
    train_time_sec: float
    peak_memory_mb: float
    final_loss: float
    after_output: str


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## Configuration 1 — Baseline

Standard attention, no gradient checkpointing. This is our reference point for both memory and speed.

In [ ]:
clear_memory()

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
if torch.cuda.is_available():
    model = model.to("cuda")
model = get_peft_model(model, build_lora_config())
load_time = time.time() - t0

t0 = time.time()
final_loss = train_loop(model, tokenizer)
train_time = time.time() - t0

peak_mem = get_gpu_peak_memory_mb()
after_output = generate_response(model, tokenizer, TEST_PROMPT)

baseline_result = SpeedResult("baseline", True, "", round(load_time, 2), round(train_time, 2),
                               round(peak_mem, 1), round(final_loss, 4), after_output)
print(baseline_result)

del model
clear_memory()

## Configuration 2 — + Gradient Checkpointing

`model.gradient_checkpointing_enable()` tells the model to discard intermediate activations during the forward pass and recompute them during backward, instead of keeping them all in memory. We also need `enable_input_require_grads()` so gradients can flow correctly through a frozen base model with LoRA adapters on top.

In [ ]:
clear_memory()

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
if torch.cuda.is_available():
    model = model.to("cuda")

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, build_lora_config())
load_time = time.time() - t0

t0 = time.time()
final_loss = train_loop(model, tokenizer)
train_time = time.time() - t0

peak_mem = get_gpu_peak_memory_mb()
after_output = generate_response(model, tokenizer, TEST_PROMPT)

gc_result = SpeedResult("gradient_checkpointing", True, "", round(load_time, 2), round(train_time, 2),
                         round(peak_mem, 1), round(final_loss, 4), after_output)
print(gc_result)

del model
clear_memory()

## Configuration 3 — + Flash Attention 2

Flash Attention 2 is a fused CUDA kernel that computes attention without ever materializing the full `(seq_len, seq_len)` attention matrix in memory — it processes it in blocks. This needs:
- An Ampere-or-newer GPU (A100, RTX 30xx/40xx, L4, etc.) — **not** available on Colab's free T4
- The `flash-attn` package installed

The cell below detects both requirements and skips cleanly with a clear message if either is missing, rather than crashing.

In [ ]:
clear_memory()
fa_result = None

if not torch.cuda.is_available():
    print("Skipped: No CUDA GPU available.")
else:
    try:
        import flash_attn  # noqa: F401
        has_flash_attn = True
    except ImportError:
        has_flash_attn = False
        print("Skipped: flash-attn not installed. Run: pip install flash-attn --no-build-isolation")

    if has_flash_attn:
        try:
            t0 = time.time()
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2"
            ).to("cuda")
            model = get_peft_model(model, build_lora_config())
            load_time = time.time() - t0

            t0 = time.time()
            final_loss = train_loop(model, tokenizer)
            train_time = time.time() - t0

            peak_mem = get_gpu_peak_memory_mb()
            after_output = generate_response(model, tokenizer, TEST_PROMPT)

            fa_result = SpeedResult("flash_attention_2", True, "", round(load_time, 2), round(train_time, 2),
                                     round(peak_mem, 1), round(final_loss, 4), after_output)
            print(fa_result)

            del model
            clear_memory()
        except Exception as e:
            print(f"Skipped: Flash Attention 2 failed to run — {e}")

if fa_result is None:
    fa_result = SpeedResult("flash_attention_2", False, "unavailable in this environment", 0, 0, 0, 0, "")

## Configuration 4 — Unsloth

Unsloth patches the underlying attention and MLP kernels with hand-optimized Triton kernels, giving noticeable training speedups with almost no code change — plus a simpler API for 4-bit loading. It also runs on T4 (unlike Flash Attention 2).

In [ ]:
clear_memory()
unsloth_result = None

if not torch.cuda.is_available():
    print("Skipped: No CUDA GPU available.")
else:
    try:
        from unsloth import FastLanguageModel
        has_unsloth = True
    except ImportError:
        has_unsloth = False
        print("Skipped: unsloth not installed. Run: pip install unsloth")

    if has_unsloth:
        try:
            t0 = time.time()
            unsloth_model, unsloth_tokenizer = FastLanguageModel.from_pretrained(
                model_name=MODEL_NAME,
                max_seq_length=128,
                dtype=None,
                load_in_4bit=True,
            )
            unsloth_model = FastLanguageModel.get_peft_model(
                unsloth_model,
                r=8,
                lora_alpha=16,
                target_modules=["q_proj", "v_proj"],
                lora_dropout=0.05,
            )
            load_time = time.time() - t0

            FastLanguageModel.for_training(unsloth_model)
            t0 = time.time()
            final_loss = train_loop(unsloth_model, unsloth_tokenizer)
            train_time = time.time() - t0

            peak_mem = get_gpu_peak_memory_mb()
            FastLanguageModel.for_inference(unsloth_model)
            after_output = generate_response(unsloth_model, unsloth_tokenizer, TEST_PROMPT)

            unsloth_result = SpeedResult("unsloth", True, "", round(load_time, 2), round(train_time, 2),
                                          round(peak_mem, 1), round(final_loss, 4), after_output)
            print(unsloth_result)

            del unsloth_model
            clear_memory()
        except Exception as e:
            print(f"Skipped: Unsloth failed to run — {e}")

if unsloth_result is None:
    unsloth_result = SpeedResult("unsloth", False, "unavailable in this environment", 0, 0, 0, 0, "")

## Side-by-side comparison

In [ ]:
all_results = [baseline_result, gc_result, fa_result, unsloth_result]

print(f"{'Config':<24}{'Ran':<6}{'Load(s)':<10}{'Train(s)':<10}{'Peak Mem(MB)':<14}{'Loss':<8}")
print("=" * 80)
for r in all_results:
    if r.ran:
        print(f"{r.config:<24}{'yes':<6}{r.load_time_sec:<10}{r.train_time_sec:<10}{r.peak_memory_mb:<14}{r.final_loss:<8}")
    else:
        print(f"{r.config:<24}{'no':<6}(skipped: {r.skip_reason})")

with open("experiment_log.json", "w") as f:
    json.dump([asdict(r) for r in all_results], f, indent=2)
print("\nSaved full results to experiment_log.json")

## What to look for

1. **Gradient checkpointing**: peak memory should drop vs baseline, but train time usually goes up slightly (the recompute cost) — a direct memory-for-compute trade.
2. **Flash Attention 2**: on a supported GPU, both memory *and* speed should improve over baseline — it's one of the few "free lunch" optimizations, but only where the hardware supports it.
3. **Unsloth**: often shows the largest speedup of the three, especially combined with its native 4-bit loading — and unlike Flash Attention 2, it also runs on older GPUs like the T4.
4. **Stacking**: in practice, these three combine — e.g. Unsloth + gradient checkpointing + QLoRA together is a very common real-world setup for training larger models on a single GPU.

## Key takeaways

- Gradient checkpointing trades compute for memory — enable it when memory is the bottleneck, not speed
- Flash Attention 2 is faster *and* lower memory, but requires Ampere+ GPU hardware and the `flash-attn` package
- Unsloth patches training with fused kernels for speedups on a wider range of GPUs (including T4), with simpler 4-bit loading built in
- These tricks stack with everything from Days 1-4 — quantization, LoRA/DoRA, and QLoRA all benefit from adding these on top

Next up: **Day 06 — Model Merging**, where we take trained LoRA adapters and merge them back into the base model for simpler deployment.